## ETAPA 6 - Evaluación del modelo de clasificación
## Objetivo

En este notebook se evaluará el rendimiento del modelo de clasificación entrenado en la fase anterior.
El objetivo es medir qué tan bien predice la vañable SANCIONADOS utilizando métricas estándar de clasificación.

## Regla de trabajo
El archivo ubicado en e: "C:\Proyecto-Risk-Score-OECE\data\processed\CONOSCE_ADJUDICACIONES_2018_2026_CONSOLIDADO_FEATURED.csv"
directamente. En esta fase se leerá el dataset se reconstruirá el flujo de preparación de datos se entrenará nuevamente el modelo base y se evaluará su desempeño con métricas de clasificación.

In [1]:
# ==========================================
# FASE 6
# EVALUACIÓN DEL MODELO DE CLASIFICACIÓN
# RISK SCORE OSCE
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ============================================================
# 1. Definir ruta del archivo de entrada
# ============================================================

ruta_featured = r"C:\Proyecto-Risk-Score-OECE\data\processed\CONOSCE_ADJUDICACIONES_2018_2026_CONSOLIDADO_FEATURED.csv"

# ============================================================
# 2. Cargar el dataset con features
# IMPORTANTE: el archivo usa punto y coma como separador
# ============================================================

df_featured = pd.read_csv(ruta_featured, sep=";")

# ============================================================
# 3. Crear una copia de trabajo
# ============================================================

df_modelo = df_featured.copy()

# ============================================================
# 4. Mostrar información inicial
# ============================================================

print("Dimensión del dataset featured:", df_modelo.shape)

print("\nColumnas disponibles:")
print(df_modelo.columns.tolist())

display(df_modelo.head())

Dimensión del dataset featured: (3875, 42)

Columnas disponibles:
['RUC', 'NOMBRE_RAZONODENOMINACIONSOCIAL', 'FECHA_INICIO', 'FECHA_FIN', 'NUMERO_RESOLUCION', 'ID_MOTIVO_INFRACCION', 'DE_MOTIVO_INFRACCION', 'motivo_grupo', 'duracion_dias', 'duracion_grupo', 'ruc_prefijo', 'fecha_segunda', 'target_sancionado_24m', 'FECHA_INICIO_DT', 'FECHA_FIN_DT', 'ANIO_INICIO_SANCION', 'MES_INICIO_SANCION', 'TRIMESTRE_INICIO_SANCION', 'PERIODO_INICIO_SANCION', 'DURACION_MESES', 'DURACION_ANIOS', 'SANCION_MAYOR_1_ANIO', 'SANCION_MAYOR_3_ANIOS', 'LOG_DURACION_DIAS', 'DURACION_DIAS_CORREGIDA', 'RUC_STR', 'RUC_PREFIJO', 'ES_PERSONA_NATURAL', 'ES_PERSONA_JURIDICA', 'ES_OTRO_TIPO_RUC', 'MOTIVO_TEXTO', 'MOTIVO_DOC_FALSA', 'MOTIVO_INFO_INEXACTA', 'MOTIVO_INCUMPLIMIENTO', 'MOTIVO_CONTRATAR_IMPEDIDO', 'NUM_MOTIVOS_INFRACCION', 'TIENE_MULTIPLES_MOTIVOS', 'TOTAL_SANCIONES_PROVEEDOR', 'ORDEN_SANCION_PROVEEDOR', 'PROVEEDOR_CON_MULTIPLES_SANCIONES', 'FECHA_PRIMERA_SANCION_PROVEEDOR', 'DIAS_DESDE_PRIMERA_SANCION']


,RUC,NOMBRE_RAZONODENOMINACIONSOCIAL,FECHA_INICIO,FECHA_FIN,NUMERO_RESOLUCION,ID_MOTIVO_INFRACCION,DE_MOTIVO_INFRACCION,motivo_grupo,duracion_dias,duracion_grupo,...,MOTIVO_INFO_INEXACTA,MOTIVO_INCUMPLIMIENTO,MOTIVO_CONTRATAR_IMPEDIDO,NUM_MOTIVOS_INFRACCION,TIENE_MULTIPLES_MOTIVOS,TOTAL_SANCIONES_PROVEEDOR,ORDEN_SANCION_PROVEEDOR,PROVEEDOR_CON_MULTIPLES_SANCIONES,FECHA_PRIMERA_SANCION_PROVEEDOR,DIAS_DESDE_PRIMERA_SANCION
0,2029124996,G & D CORPORACION DE NEGOCIOS LACTEOS S.A. COR...,20050608,NaN,508-2005-TC-SU,8,DOCUMENTOS FALSOS,Documentación falsa/inexacta,NaN,Sin fecha fin,...,0,0,0,1,0,1,1,0,2005-06-08,0
1,10000282125,GOMEZ CASTRO JUANA,20200616,20230716.0,1110-2020-TCE-S2,"215,",Presentar documentos falsos o adulterados a la...,Documentación falsa/inexacta,1125.0,Más de 3 años,...,0,0,0,1,0,1,1,0,2020-06-16,0
2,10000710623,OLIVEIRA DE MACHUCA LITA,20191107,20230207.0,2903-2019-TCE-S2,"214,215,",Presentar información inexacta a las Entidades...,Documentación falsa/inexacta,1188.0,Más de 3 años,...,1,0,0,2,1,1,1,0,2019-11-07,0
3,10001253609,AGUIRRE VARGAS GUSTAVO ALFREDO,20190807,20220907.0,2147-2019-TCE-S3,"214,215,",Presentar información inexacta a las Entidades...,Documentación falsa/inexacta,1127.0,Más de 3 años,...,1,0,0,2,1,1,1,0,2019-08-07,0
4,10002101756,MENDOZA PENA JULIO CESAR,20231005,20240205.0,3820-2023-TCE-S6,"241,",f) Ocasionar que la Entidad resuelva el contra...,Resolución/rescisión contractual,123.0,Hasta 6 meses,...,0,0,0,1,0,1,1,0,2023-10-05,0


In [2]:
# ============================================================
# 5. Definir variable objetivo
# ============================================================

target = "target_sancionado_24m"

print("Variable objetivo:", target)

print("\nDistribución de clases:")
print(df_modelo[target].value_counts())

print("\nDistribución porcentual:")
print((df_modelo[target].value_counts(normalize=True) * 100).round(2))

# ============================================================
# 6. Seleccionar variables predictoras iniciales
# ============================================================

columnas_modelo = [
    # Variables temporales
    "ANIO_INICIO_SANCION",
    "MES_INICIO_SANCION",
    "TRIMESTRE_INICIO_SANCION",

    # Variables de duración / severidad
    "DURACION_MESES",
    "DURACION_ANIOS",
    "SANCION_MAYOR_1_ANIO",
    "SANCION_MAYOR_3_ANIOS",
    "LOG_DURACION_DIAS",

    # Variables del tipo de proveedor
    "ES_PERSONA_NATURAL",
    "ES_PERSONA_JURIDICA",
    "ES_OTRO_TIPO_RUC",

    # Variables del motivo de infracción
    "MOTIVO_DOC_FALSA",
    "MOTIVO_INFO_INEXACTA",
    "MOTIVO_INCUMPLIMIENTO",
    "MOTIVO_CONTRATAR_IMPEDIDO",
    "NUM_MOTIVOS_INFRACCION",
    "TIENE_MULTIPLES_MOTIVOS",

    # Variables categóricas
    "motivo_grupo",
    "duracion_grupo"
]

# Validar que las columnas existan
columnas_existentes = [col for col in columnas_modelo if col in df_modelo.columns]
columnas_faltantes = [col for col in columnas_modelo if col not in df_modelo.columns]

print("\nColumnas usadas en el modelo:")
print(columnas_existentes)

print("\nColumnas faltantes que no se usarán:")
print(columnas_faltantes)

df_modelo = df_modelo[columnas_existentes + [target]].copy()

print("\nDimensión del dataset para modelamiento:")
print(df_modelo.shape)

display(df_modelo.head())

Variable objetivo: target_sancionado_24m

Distribución de clases:
target_sancionado_24m
0    3139
1     736
Name: count, dtype: int64

Distribución porcentual:
target_sancionado_24m
0    81.01
1    18.99
Name: proportion, dtype: float64

Columnas usadas en el modelo:
['ANIO_INICIO_SANCION', 'MES_INICIO_SANCION', 'TRIMESTRE_INICIO_SANCION', 'DURACION_MESES', 'DURACION_ANIOS', 'SANCION_MAYOR_1_ANIO', 'SANCION_MAYOR_3_ANIOS', 'LOG_DURACION_DIAS', 'ES_PERSONA_NATURAL', 'ES_PERSONA_JURIDICA', 'ES_OTRO_TIPO_RUC', 'MOTIVO_DOC_FALSA', 'MOTIVO_INFO_INEXACTA', 'MOTIVO_INCUMPLIMIENTO', 'MOTIVO_CONTRATAR_IMPEDIDO', 'NUM_MOTIVOS_INFRACCION', 'TIENE_MULTIPLES_MOTIVOS', 'motivo_grupo', 'duracion_grupo']

Columnas faltantes que no se usarán:
[]

Dimensión del dataset para modelamiento:
(3875, 20)


,ANIO_INICIO_SANCION,MES_INICIO_SANCION,TRIMESTRE_INICIO_SANCION,DURACION_MESES,DURACION_ANIOS,SANCION_MAYOR_1_ANIO,SANCION_MAYOR_3_ANIOS,LOG_DURACION_DIAS,ES_PERSONA_NATURAL,ES_PERSONA_JURIDICA,ES_OTRO_TIPO_RUC,MOTIVO_DOC_FALSA,MOTIVO_INFO_INEXACTA,MOTIVO_INCUMPLIMIENTO,MOTIVO_CONTRATAR_IMPEDIDO,NUM_MOTIVOS_INFRACCION,TIENE_MULTIPLES_MOTIVOS,motivo_grupo,duracion_grupo,target_sancionado_24m
0,2005,6,2,NaN,NaN,0,0,NaN,0,0,1,1,0,0,0,1,0,Documentación falsa/inexacta,Sin fecha fin,0
1,2020,6,2,37.500000,3.082192,1,1,7.026427,1,0,0,1,0,0,0,1,0,Documentación falsa/inexacta,Más de 3 años,0
2,2019,11,4,39.600000,3.254795,1,1,7.080868,1,0,0,1,1,0,0,2,1,Documentación falsa/inexacta,Más de 3 años,0
3,2019,8,3,37.566667,3.087671,1,1,7.028201,1,0,0,1,1,0,0,2,1,Documentación falsa/inexacta,Más de 3 años,0
4,2023,10,4,4.100000,0.336986,0,0,4.820282,1,0,0,0,0,0,0,1,0,Resolución/rescisión contractual,Hasta 6 meses,0


In [3]:
# ============================================================
# 7. Limpieza básica antes de evaluar
# ============================================================

# Reemplazar valores infinitos por NaN
df_modelo = df_modelo.replace([np.inf, -np.inf], np.nan)

print("Nulos antes del tratamiento:")
print(df_modelo.isnull().sum())

# Separar columnas numéricas y categóricas
columnas_numericas = df_modelo.select_dtypes(include=["int64", "float64"]).columns.tolist()
columnas_categoricas = df_modelo.select_dtypes(include=["object"]).columns.tolist()

# Quitar target si aparece entre numéricas
if target in columnas_numericas:
    columnas_numericas.remove(target)

# Rellenar nulos numéricos con mediana
for col in columnas_numericas:
    df_modelo[col] = df_modelo[col].fillna(df_modelo[col].median())

# Rellenar nulos categóricos con SIN_DATO
for col in columnas_categoricas:
    df_modelo[col] = df_modelo[col].fillna("SIN_DATO")

print("\nNulos después del tratamiento:")
print(df_modelo.isnull().sum())

Nulos antes del tratamiento:
ANIO_INICIO_SANCION            0
MES_INICIO_SANCION             0
TRIMESTRE_INICIO_SANCION       0
DURACION_MESES               528
DURACION_ANIOS               528
SANCION_MAYOR_1_ANIO           0
SANCION_MAYOR_3_ANIOS          0
LOG_DURACION_DIAS            539
ES_PERSONA_NATURAL             0
ES_PERSONA_JURIDICA            0
ES_OTRO_TIPO_RUC               0
MOTIVO_DOC_FALSA               0
MOTIVO_INFO_INEXACTA           0
MOTIVO_INCUMPLIMIENTO          0
MOTIVO_CONTRATAR_IMPEDIDO      0
NUM_MOTIVOS_INFRACCION         0
TIENE_MULTIPLES_MOTIVOS        0
motivo_grupo                   0
duracion_grupo                 0
target_sancionado_24m          0
dtype: int64

Nulos después del tratamiento:
ANIO_INICIO_SANCION          0
MES_INICIO_SANCION           0
TRIMESTRE_INICIO_SANCION     0
DURACION_MESES               0
DURACION_ANIOS               0
SANCION_MAYOR_1_ANIO         0
SANCION_MAYOR_3_ANIOS        0
LOG_DURACION_DIAS            0
ES_PERSONA_NATURAL

In [4]:
# ============================================================
# 8. Separar variables predictoras (X) y variable objetivo (y)
# ============================================================

X = df_modelo.drop(columns=[target])
y = df_modelo[target]

print("Dimensión de X antes de get_dummies:", X.shape)
print("Dimensión de y:", y.shape)

# ============================================================
# 9. Convertir variables categóricas a variables numéricas
# ============================================================

X = pd.get_dummies(X, drop_first=True)

print("\nDimensión de X después de get_dummies:")
print(X.shape)

# ============================================================
# 10. Dividir en entrenamiento y prueba con estratificación
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nDimensión de X_train:", X_train.shape)
print("Dimensión de X_test:", X_test.shape)
print("Dimensión de y_train:", y_train.shape)
print("Dimensión de y_test:", y_test.shape)

print("\nDistribución de clases en y_train:")
print(y_train.value_counts(normalize=True).round(4) * 100)

print("\nDistribución de clases en y_test:")
print(y_test.value_counts(normalize=True).round(4) * 100)

Dimensión de X antes de get_dummies: (3875, 19)
Dimensión de y: (3875,)

Dimensión de X después de get_dummies:
(3875, 26)

Dimensión de X_train: (3100, 26)
Dimensión de X_test: (775, 26)
Dimensión de y_train: (3100,)
Dimensión de y_test: (775,)

Distribución de clases en y_train:
target_sancionado_24m
0    81.0
1    19.0
Name: proportion, dtype: float64

Distribución de clases en y_test:
target_sancionado_24m
0    81.03
1    18.97
Name: proportion, dtype: float64


In [5]:
# ============================================================
# 11. Crear modelo corregido para desbalance
# ============================================================

modelo = LogisticRegression(
    max_iter=20000,
    random_state=42,
    class_weight="balanced"
)

# ============================================================
# 12. Entrenar el modelo
# ============================================================

modelo.fit(X_train, y_train)

print("Modelo entrenado correctamente.")
print("Clases detectadas por el modelo:")
print(modelo.classes_)

Modelo entrenado correctamente.
Clases detectadas por el modelo:
[0 1]


In [6]:
# ============================================================
# 13. Generar predicciones
# ============================================================

y_pred = modelo.predict(X_test)

# Probabilidad de clase positiva = 1
y_prob = modelo.predict_proba(X_test)[:, 1]

# ============================================================
# 14. Calcular métricas
# ============================================================

accuracy = accuracy_score(y_test, y_pred)
matriz_confusion = confusion_matrix(y_test, y_pred)
reporte_clasificacion = classification_report(y_test, y_pred)

precision_clase_1 = precision_score(y_test, y_pred, pos_label=1)
recall_clase_1 = recall_score(y_test, y_pred, pos_label=1)
f1_clase_1 = f1_score(y_test, y_pred, pos_label=1)
roc_auc = roc_auc_score(y_test, y_prob)

# ============================================================
# 15. Mostrar resultados
# ============================================================

print("\n" + "="*60)
print("RESULTADOS DE EVALUACIÓN - RISK SCORE OSCE")
print("="*60)

print(f"\nAccuracy del modelo: {accuracy:.4f}")

print("\nMatriz de confusión:")
print(matriz_confusion)

print("\nReporte de clasificación:")
print(reporte_clasificacion)

print("\nMétricas específicas para la clase positiva 1:")
print(f"Precision clase 1: {precision_clase_1:.4f}")
print(f"Recall clase 1:    {recall_clase_1:.4f}")
print(f"F1-score clase 1:  {f1_clase_1:.4f}")
print(f"ROC-AUC:           {roc_auc:.4f}")


RESULTADOS DE EVALUACIÓN - RISK SCORE OSCE

Accuracy del modelo: 0.7135

Matriz de confusión:
[[483 145]
 [ 77  70]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.86      0.77      0.81       628
           1       0.33      0.48      0.39       147

    accuracy                           0.71       775
   macro avg       0.59      0.62      0.60       775
weighted avg       0.76      0.71      0.73       775


Métricas específicas para la clase positiva 1:
Precision clase 1: 0.3256
Recall clase 1:    0.4762
F1-score clase 1:  0.3867
ROC-AUC:           0.6577


In [7]:
# ============================================================
# 16. Crear tabla de resultados con identificación del proveedor
# ============================================================

indices_test = X_test.index

columnas_identificacion = [
    "RUC",
    "NOMBRE_RAZONODENOMINACIONSOCIAL",
    "FECHA_INICIO",
    "FECHA_FIN",
    "motivo_grupo",
    "duracion_grupo"
]

columnas_identificacion = [
    col for col in columnas_identificacion if col in df_featured.columns
]

resultados = df_featured.loc[indices_test, columnas_identificacion].copy()

resultados["target_real"] = y_test.values
resultados["prediccion_modelo"] = y_pred
resultados["probabilidad_sancion_24m"] = y_prob

resultados["nivel_riesgo"] = pd.cut(
    resultados["probabilidad_sancion_24m"],
    bins=[0, 0.30, 0.60, 1.00],
    labels=["Bajo", "Medio", "Alto"],
    include_lowest=True
)

resultados = resultados.sort_values(
    by="probabilidad_sancion_24m",
    ascending=False
)

display(resultados.head(20))

,RUC,NOMBRE_RAZONODENOMINACIONSOCIAL,FECHA_INICIO,FECHA_FIN,motivo_grupo,duracion_grupo,target_real,prediccion_modelo,probabilidad_sancion_24m,nivel_riesgo
1731,20489091675,SISA TOURS SAC,20240429,20240729.0,Impedimento para contratar,Hasta 6 meses,0,1,0.833974,Alto
2325,20527015627,SERVICENTRO EL PORVENIR E.I.R.L. - SERVICENTRO...,20230711,20231011.0,Impedimento para contratar,Hasta 6 meses,1,1,0.832107,Alto
3303,20600738179,PERUANA DE SERVICIOS INTEGRALES S.A.C.,20230213,20230513.0,Impedimento para contratar,Hasta 6 meses,1,1,0.828399,Alto
1528,20480052090,NEGOCIOS & CONSTRUCCIONES LITO E.I.R.L.,20221010,20230110.0,Documentación falsa/inexacta,Hasta 6 meses,0,1,0.826262,Alto
3194,20600192613,FERSCONS E.I.R.L.,20240522,20240822.0,Impedimento para contratar,Hasta 6 meses,1,1,0.825499,Alto
2441,20531497156,G&M NETWORK E.I.R.L.,20240325,20240625.0,Impedimento para contratar,Hasta 6 meses,1,1,0.818459,Alto
1077,20352428648,J.P.C. INGENIEROS S.A.C.,20220720,20220822.0,No perfecciona/suscribe contrato,Hasta 6 meses,0,1,0.817975,Alto
2438,20531345305,RADIO MASTER EIRL,20240619,20240919.0,Impedimento para contratar,Hasta 6 meses,0,1,0.817413,Alto
1234,20414235108,SCHNEIDER ELECTRIC SYSTEMS DEL PERU S.A.,20220603,20220903.0,Impedimento para contratar,Hasta 6 meses,0,1,0.816951,Alto
123,10086163662,ESPINOZA RODAS GUILLERMO,20240116,20240416.0,Impedimento para contratar,Hasta 6 meses,0,1,0.816910,Alto


In [8]:
# ============================================================
# 17. Guardar predicciones evaluadas
# ============================================================

ruta_predicciones_evaluacion = r"C:\Proyecto-Risk-Score-OECE\data\processed\predicciones_evaluacion_riskscore_osce.csv"

resultados.to_csv(
    ruta_predicciones_evaluacion,
    sep=";",
    index=True,
    encoding="utf-8-sig"
)

print("Archivo de predicciones de evaluación guardado en:")
print(ruta_predicciones_evaluacion)

Archivo de predicciones de evaluación guardado en:
C:\Proyecto-Risk-Score-OECE\data\processed\predicciones_evaluacion_riskscore_osce.csv


## ETAPA 6.1 - Evaluación comparativa con Random Forest balanceado

### Objetivo

En esta sección se evaluará un modelo alternativo basado en Random Forest para comparar su desempeño frente a la Regresión Logística balanceada.

### Justificación metodológica

Random Forest puede capturar relaciones no lineales entre variables de duración, motivo de infracción y tipo de proveedor. Además, se utilizará `class_weight="balanced"` para reducir el sesgo frente al desbalance de clases.

In [9]:
# ======================================================
# FASE 6.1
# EVALUACIÓN CON RANDOM FOREST BALANCEADO
# ======================================================

from sklearn.ensemble import RandomForestClassifier

# ======================================================
# 1. Crear modelo Random Forest balanceado
# ======================================================

modelo_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# ======================================================
# 2. Entrenar modelo
# ======================================================

modelo_rf.fit(X_train, y_train)

# ======================================================
# 3. Generar predicciones
# ======================================================

y_pred_rf = modelo_rf.predict(X_test)
y_prob_rf = modelo_rf.predict_proba(X_test)[:, 1]

# ======================================================
# 4. Calcular métricas
# ======================================================

accuracy_rf = accuracy_score(y_test, y_pred_rf)
matriz_rf = confusion_matrix(y_test, y_pred_rf)
reporte_rf = classification_report(y_test, y_pred_rf)

precision_rf_1 = precision_score(y_test, y_pred_rf, pos_label=1)
recall_rf_1 = recall_score(y_test, y_pred_rf, pos_label=1)
f1_rf_1 = f1_score(y_test, y_pred_rf, pos_label=1)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

# ======================================================
# 5. Mostrar resultados
# ======================================================

print("\n" + "="*60)
print("RESULTADOS RANDOM FOREST BALANCEADO - RISK SCORE OSCE")
print("="*60)

print(f"\nAccuracy Random Forest: {accuracy_rf:.4f}")

print("\nMatriz de confusión:")
print(matriz_rf)

print("\nReporte de clasificación:")
print(reporte_rf)

print("\nMétricas específicas para clase positiva 1:")
print(f"Precision clase 1 RF: {precision_rf_1:.4f}")
print(f"Recall clase 1 RF:    {recall_rf_1:.4f}")
print(f"F1-score clase 1 RF:  {f1_rf_1:.4f}")
print(f"ROC-AUC RF:           {roc_auc_rf:.4f}")


RESULTADOS RANDOM FOREST BALANCEADO - RISK SCORE OSCE

Accuracy Random Forest: 0.7071

Matriz de confusión:
[[490 138]
 [ 89  58]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.85      0.78      0.81       628
           1       0.30      0.39      0.34       147

    accuracy                           0.71       775
   macro avg       0.57      0.59      0.58       775
weighted avg       0.74      0.71      0.72       775


Métricas específicas para clase positiva 1:
Precision clase 1 RF: 0.2959
Recall clase 1 RF:    0.3946
F1-score clase 1 RF:  0.3382
ROC-AUC RF:           0.6579


In [10]:
# ======================================================
# 18. Comparación de modelos
# ======================================================

comparacion_modelos = pd.DataFrame({
    "Modelo": [
        "Regresión Logística balanceada",
        "Random Forest balanceado"
    ],
    "Accuracy": [
        accuracy,
        accuracy_rf
    ],
    "Precision_clase_1": [
        precision_clase_1,
        precision_rf_1
    ],
    "Recall_clase_1": [
        recall_clase_1,
        recall_rf_1
    ],
    "F1_clase_1": [
        f1_clase_1,
        f1_rf_1
    ],
    "ROC_AUC": [
        roc_auc,
        roc_auc_rf
    ]
})

display(comparacion_modelos)

,Modelo,Accuracy,Precision_clase_1,Recall_clase_1,F1_clase_1,ROC_AUC
0,Regresión Logística balanceada,0.713548,0.325581,0.476190,0.386740,0.657660
1,Random Forest balanceado,0.707097,0.295918,0.394558,0.338192,0.657876


In [11]:
ruta_comparacion = r"C:\Proyecto-Risk-Score-OECE\data\processed\comparacion_modelos_riskscore_osce.csv"

comparacion_modelos.to_csv(
    ruta_comparacion,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Comparación de modelos guardada en:")
print(ruta_comparacion)

Comparación de modelos guardada en:
C:\Proyecto-Risk-Score-OECE\data\processed\comparacion_modelos_riskscore_osce.csv


## ETAPA 6.2 - Análisis de importancia de variables

### Objetivo

En esta sección se analizará qué variables tienen mayor peso en la predicción del riesgo de sanción futura del proveedor.

### Justificación

Además de evaluar qué modelo predice mejor, el proyecto RiskScore OSCE requiere explicar qué factores contribuyen al score de riesgo. Para ello, se utilizará la importancia de variables del modelo Random Forest.

In [12]:
# ======================================================
# FASE 6.2
# ANÁLISIS DE IMPORTANCIA DE VARIABLES - RANDOM FOREST
# ======================================================

import pandas as pd

# Crear tabla con importancia de variables
importancia_variables = pd.DataFrame({
    "variable": X.columns,
    "importancia": modelo_rf.feature_importances_
})

# Ordenar de mayor a menor importancia
importancia_variables = importancia_variables.sort_values(
    by="importancia",
    ascending=False
)

print("Top 20 variables más importantes para el modelo Random Forest:")
display(importancia_variables.head(20))

Top 20 variables más importantes para el modelo Random Forest:


,variable,importancia
1,MES_INICIO_SANCION,0.141605
0,ANIO_INICIO_SANCION,0.141243
4,DURACION_ANIOS,0.120643
7,LOG_DURACION_DIAS,0.118082
3,DURACION_MESES,0.114558
2,TRIMESTRE_INICIO_SANCION,0.066083
14,MOTIVO_CONTRATAR_IMPEDIDO,0.049540
12,MOTIVO_INFO_INEXACTA,0.034508
8,ES_PERSONA_NATURAL,0.028142
9,ES_PERSONA_JURIDICA,0.024051


In [13]:
# ======================================================
# GUARDAR IMPORTANCIA DE VARIABLES
# ======================================================

ruta_importancias = r"C:\Proyecto-Risk-Score-OECE\data\processed\importancia_variables_random_forest_riskscore_osce.csv"

importancia_variables.to_csv(
    ruta_importancias,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo de importancia de variables guardado en:")
print(ruta_importancias)

Archivo de importancia de variables guardado en:
C:\Proyecto-Risk-Score-OECE\data\processed\importancia_variables_random_forest_riskscore_osce.csv
